In [ ]:
# Task 1

In [ ]:
pip install ucimlrepo

In [ ]:
# FASE PRELIMINARE 1 - Installazione della libreria esterna UCI Machine Learning Repository


# Importazione del dataset scelto "Car Evaluation" che ha ID = 19
from ucimlrepo import fetch_ucirepo 
car_evaluation = fetch_ucirepo(id=19) 
  
# Varibile che contiene le caratteristiche degli oggetti
X = car_evaluation.data.features 

# Varibile che contiene l'etichetta finale, la classificazione che si vuole prevedere
y = car_evaluation.data.targets 
  
# Visualizzazione dei metadata 
print(car_evaluation.metadata)   

# Visualizzazione delle varibili
print(car_evaluation.variables) 


In [ ]:
#FASE PRELIMINARE 2 Utilizzo della Libreria Pandas (ideale per lavorare con dati tabulari)
import pandas as pd

# Imposto la visualizzazione del dataset a 10 righe (per comodità di visualizzazione)
pd.set_option('display.max_rows', 10)      
  
# Concateno le caratteristiche (X) con i target (y)
dataset_completo = pd.concat([X, y], axis=1)

# Visualizzo il dataset_completo 
print("--- DATASET 'CAR EVALUATION' ---")
print(dataset_completo)

In [ ]:
# FASE 1 - ENCODING (Conversione Testo -> Numeri)

# Si crea una mappatura per convertire correttamente ogni stringa (etichetta) in un numero ordinale
mappatura = {
    'buying':   {'low': 0, 'med': 1, 'high': 2, 'vhigh': 3},
    'maint':    {'low': 0, 'med': 1, 'high': 2, 'vhigh': 3},
    'doors':    {'2': 2, '3': 3, '4': 4, '5more': 5},
    'persons':  {'2': 2, '4': 4, 'more': 5},
    'lug_boot': {'small': 0, 'med': 1, 'big': 2},
    'safety':   {'low': 0, 'med': 1, 'high': 2}
}

# Creazione di una nuova variabile dove si sotituiscono i vecchi valori "stringa" da X con i nuovi valori "interi"
Xnuovo = X.replace(mappatura)

print(Xnuovo)

In [ ]:
# FASE 2 - STANDARDIZZAZIONE (la PCA richiede dati centrati (media 0, deviazione standard 1))
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_standardizzata = scaler.fit_transform(Xnuovo)

print(X_standardizzata)

In [ ]:
# FASE 3 - APPLICAZIONE DELLA PCA (si passa dalle 6 dimensioni originali a sole 2 dimensioni (Componenti Principali))
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_standardizzata)

print(X_pca)

In [ ]:
# FASE 3 APPROFONDIMENTO - ESTRAZIONE DEI COEFFICIENTI (AUTOVETTORI) (per comprendere il peso dei singoli componenti nei due nuovi componenti principali)
import seaborn as sns
import matplotlib.pyplot as plt

pesi_componenti = pca.components_

df_loadings = pd.DataFrame(
    data=pesi_componenti, 
    columns=Xnuovo.columns, 
    index=['PC1 (Asse X)', 'PC2 (Asse Y)']
).T 

plt.figure(figsize=(6, 5))
sns.heatmap(df_loadings, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f")
plt.title("Contributo delle Feature su PC1 e PC2 (Loadings)")
plt.ylabel("Caratteristiche Originali")
plt.show()


import numpy as np
varianza_individuale = pca.explained_variance_ratio_
varianza_cumulativa = np.sum(varianza_individuale)

# Visualizziamo la varianza spiegata dalla PC1 e PC2 rispetto alle 6 componenti iniziali
print(f"Varianza spiegata dalla PC1: {varianza_individuale[0]:.2%}")
print(f"Varianza spiegata dalla PC2: {varianza_individuale[1]:.2%}")
print(f"Varianza Cumulativa (Totale informazione conservata): {varianza_cumulativa:.2%}")

In [ ]:
# FASE 4 - VISUALIZZAZIONE DEI RISULTATI (Scatter Plot)
sns.set_theme(style="whitegrid")

# Creazione di un nuovo DataFrame Pandas per la visualizzazione 
df_pca = pd.DataFrame(data=X_pca, columns=['PC1', 'PC2'])
df_pca['Target'] = y['class'].values 

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='Target', palette='viridis', s=70)
plt.title("PCA del dataset Car Evaluation (Visualizzazione 2D)")
plt.xlabel("Componente Principale 1 (PC1)")
plt.ylabel("Componente Principale 2 (PC2)")
plt.legend(title='Valutazione Auto')
plt.show()

In [ ]:
#Task 2

In [ ]:
# Fase 1 - SUDDIVISIONE DEI DATI (70% Train, 30% Test)
from sklearn.model_selection import train_test_split

# Fase 1.1 - Suddivisione in 70% Train e 30% Temp
X_train, X_temp, y_train, y_temp = train_test_split(Xnuovo, y['class'], test_size=0.3, random_state=42, stratify=y)

# Fase 1.2 Suddivisione il 30% rimanente temp in 20% validation e 10% test
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.33, random_state=42, stratify=y_temp)

print(f"Totale set: {len(X)} campioni");


print(f"Training set: {len(X_train)} campioni");
print(f"Validation set: {len(X_valid)} campioni");
print(f"Test set: {len(X_test)} campioni");



In [ ]:
#Fase 2 - STANDARDIZZAZIONE DEI DATI
from sklearn.preprocessing import StandardScaler

# La standardizzazione va fatta esclusivamente sul training set (in modo da non "vedere" i dati del test set)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Fase 3 - INIZIALIZZAZIONE DEI MODELLI

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

# Modello 1: Support Vector Machine (kernel RBF di default)
modello_svm = SVC(kernel='rbf', C=1.0, gamma='auto', probability=True)

# Modello 2: Random Forest (100 alberi decisionali)
modello_rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Modello 3: K-Nearest Neighbors
# Eseguiamo prima la CROSS-VALIDATION per trovare il K migliore da usare nel modello.

k_range = range (1,26)
k_scores = []

print("Ricerca del K ottimale tramite Cross-Validation...")
for k in k_range:
    knn_cv = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_cv, X_train_scaled, y_train, cv=10, scoring='accuracy')
    k_scores.append(scores.mean())

optimal_k = k_range[np.argmax(k_scores)]
print(f"Il valore ottimale di K è:{optimal_k}\n")

# Ifine inizializziamo il modello con il K migliore trovato
modello_knn = KNeighborsClassifier(n_neighbors=optimal_k, weights='distance', metric='euclidean')

In [ ]:
# Fase 4 - ADDESTRAMENTO (FITTING)

print("Addestramento dei modelli in corso...")
modello_svm.fit(X_train_scaled, y_train)
modello_rf.fit(X_train_scaled, y_train)
modello_knn.fit(X_train_scaled, y_train)
print("Addestramento completato!")

In [ ]:
# Fase 5 - PREDIZIONE SUL TEST SET

y_pred_svm = modello_svm.predict(X_test_scaled)
y_pred_rf = modello_rf.predict(X_test_scaled)
y_pred_knn = modello_knn.predict(X_test_scaled)

print("Predizioni generate e pronte per la valutazione")

In [ ]:
# Task 3 - Implementazione del calcolo delle metriche di valutazione precision, recall, f-measure Accuracy, Roc AUC

In [ ]:
# Fase 1 - VALUTAZIONE DEI MODELLI

from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# 1 - Modello SVM
print("Valutazione modello SVM")
# Accuracy
accuracy_modello_svm = accuracy_score(y_test, y_pred_svm)
print(f"Accuracy SVM: {accuracy_modello_svm:.4f}")

# Calcolo di Precision, Recall, F-Measure per ciascuna classe
print("Report di Classificazione (Dettaglio per classe) SVM:")
print(classification_report(y_test, y_pred_svm))

# Calcolo e stampa del ROC AUC Score 
y_proba_svm = modello_svm.predict_proba(X_test_scaled)
roc_auc_svm = roc_auc_score(y_test, y_proba_svm, multi_class='ovr', average='macro')
print(f"ROC AUC Score (Macro) SVM: {roc_auc_svm:.4f}")

print("")


# 2 - Modello Random Forest
print("Valutazione modello Random Forest")
# Accuracy
accuracy_modello_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuracy RF: {accuracy_modello_rf:.4f}")

# Calcolo di Precision, Recall, F-Measure per ciascuna classe
print("Report di Classificazione (Dettaglio per classe) RF:")
print(classification_report(y_test, y_pred_rf))

# Calcolo e stampa del ROC AUC Score 
y_proba_rf = modello_rf.predict_proba(X_test_scaled)
roc_auc_rf = roc_auc_score(y_test, y_proba_rf, multi_class='ovr', average='macro')
print(f"ROC AUC Score (Macro) RF: {roc_auc_rf:.4f}")

print("")


# 3 - Modello KNN
print("Valutazione modello KNN")

# Accuracy
accuracy_modello_knn = accuracy_score(y_test, y_pred_knn)
print(f"Accuracy KNN: {accuracy_modello_knn:.4f}")

# Calcolo di Precision, Recall, F-Measure per ciascuna classe
print("Report di Classificazione (Dettaglio per classe) KNN:")
print(classification_report(y_test, y_pred_knn))

# Calcolo e stampa del ROC AUC Score 
y_proba_knn = modello_knn.predict_proba(X_test_scaled)
roc_auc_knn = roc_auc_score(y_test, y_proba_knn, multi_class='ovr', average='macro')
print(f"ROC AUC Score (Macro) KNN: {roc_auc_knn:.4f}")


In [ ]:
# Task 4 - Implementazione della visualizzazione della matrice di confusione e della ROC AUC Curve

In [ ]:
# Fase 1 

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

classi_uniche = y['class'].unique()

# Matrice di Confusione per SVM
confusionMatrixSVM = confusion_matrix(y_test, y_pred_svm)
dispSVM = ConfusionMatrixDisplay(confusion_matrix=confusionMatrixSVM, display_labels=classi_uniche)
dispSVM.plot(cmap='Blues')
plt.title("Matrice di Confusione - SVM (RBF Kernel)")
plt.show()

# Matrice di Confusione per RF
confusionMatrixRF = confusion_matrix(y_test, y_pred_rf)
dispRF = ConfusionMatrixDisplay(confusion_matrix=confusionMatrixRF, display_labels=classi_uniche)
dispRF.plot(cmap='Reds')
plt.title("Matrice di Confusione - RF")
plt.show()

# Matrice di Confusione per KNN
confusionMatrixKNN = confusion_matrix(y_test, y_pred_knn)
dispKNN = ConfusionMatrixDisplay(confusion_matrix=confusionMatrixKNN, display_labels=classi_uniche)
dispKNN.plot(cmap='Greens')
plt.title("Matrice di Confusione - KNN")
plt.show()

In [ ]:
# Fase 2 - CURVE ROC 
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

# Calcoliamo le probabilità predette per ogni modello sui dati scalati
y_proba_svm = modello_svm.predict_proba(X_test_scaled)
y_proba_rf = modello_rf.predict_proba(X_test_scaled)
y_proba_knn = modello_knn.predict_proba(X_test_scaled)

classi_uniche = y['class'].unique()
y_test_bin = label_binarize(y_test, classes=classi_uniche)
n_classes = len(classi_uniche)

modelli = {
    "Support Vector Machine": y_proba_svm,
    "Random Forest": y_proba_rf,
    "K-Nearest Neighbors": y_proba_knn
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confronto Curve ROC (One-vs-Rest) - Modelli Classici', fontsize=16, fontweight='bold')

for idx, (nome_modello, y_proba) in enumerate(modelli.items()):
    ax = axes[idx]
    
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=2, label=f'{classi_uniche[i]} (AUC = {roc_auc:.2f})')
    
    ax.plot([0, 1], [0, 1], color='gray', linestyle='--') # Linea casuale
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_title(nome_modello, fontsize=12)
    ax.set_xlabel('False Positive Rate')
    if idx == 0:
        ax.set_ylabel('True Positive Rate')
    ax.legend(loc="lower right")
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Task 5 - Implementazione di un approccio di deep learning sugli stessi dati utilizzati per gli algoritmi shallow

In [ ]:
!pip install tensorboard

In [ ]:
# Fase 1 - Preparazione target per rete neurale

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print("TensorFlow versione:", tf.__version__)

In [ ]:
# Fase 2 - costruzione del modello di rete neurale
# Inizializziamo la PCA a 2 componenti
pca = PCA(n_components=2)

# Apprendiamo e trasformiamo SOLO il training set (evita il Data Leakage)
X_train_pca = pca.fit_transform(X_train_scaled)
X_valid_scaled = scaler.transform(X_valid)

X_test_pca = pca.transform(X_test_scaled)
X_valid_pca = pca.transform(X_valid_scaled)

print("PCA applicata con successo: dimensioni ridotte a", X_train_pca.shape[1], "componenti.\n")


model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(2,)),
    keras.layers.Dense(4, activation='softmax')
])

model.summary()

In [ ]:
# Fase 3 - Compilazione del Modello
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_num = le.fit_transform(y_train)


y_valid_num = le.transform(y_valid)
y_test_num = le.transform(y_test)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', 
    metrics=['accuracy']
)

In [ ]:
# Fase 4 - Addestramento del modello con attivazione di tensorboard
log_dir = "logs/intro_dl"
tensorboard_cb = keras.callbacks.TensorBoard(log_dir=log_dir)

print("Addestramento Rete Neurale in corso...")

history = model.fit(
    X_train_pca, y_train_num,
    validation_data=(X_valid_pca, y_valid_num), 
    epochs=100,
    batch_size=16,
    callbacks=[tensorboard_cb]
)

print("\nValutazione sul Test Set:")
loss, acc = model.evaluate(X_test_pca, y_test_num, verbose=0)
print(f"Accuracy della Rete Neurale: {acc:.4f}")

In [ ]:
# Fase 5. Valutazione e visualizzazione grafico sul test set
loss, acc = model.evaluate(X_test_pca, y_test_num, verbose=0)
print(f"\nAccuratezza della Rete Neurale sul Test Set: {acc:.4f}")

plt.plot(history.history['loss'])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

In [ ]:
# Fase 6 - Valutazione dettagliata rete neurale 

# Si prende l'indice con la probabilità più alta.
y_pred_proba = model.predict(X_test_pca)
y_pred_dl = np.argmax(y_pred_proba, axis=1)

# Classification Report Dettagliato
print("\n" + "="*60)
print("REPORT DI CLASSIFICAZIONE - RETE NEURALE (MLP)")
print("="*60)

print(classification_report(y_test_num, y_pred_dl, target_names=le.classes_, zero_division=0))

cm_dl = confusion_matrix(y_test_num, y_pred_dl)
disp_dl = ConfusionMatrixDisplay(confusion_matrix=cm_dl, display_labels=le.classes_)

disp_dl.plot(cmap='Purples')

plt.title("Matrice di Confusione - Rete Neurale (PCA 2D)", fontsize=12, fontweight='bold')
plt.grid(False)
plt.show()